# Module 11: BERT document embeddings for MD&A text

Re-runs Module 10's GBM with 768 BERT embedding columns appended to the ratio/industry feature set, to see whether MD&A text adds predictive power over financials alone.

**Encoding runs on the WRDS grid, not inline in this notebook.** BERT-encoding ~6,500 documents (chunked at 510 tokens each) is too heavy for interactive use. The pipeline is:

1. `src/text/download_model.py` — run once, manually, on the WRDS **login node** (has internet) to cache `bert-base-uncased` under `$HF_HOME` on scratch.
2. `jobs/encode_mdna.sh` — a 20-task array job on the WRDS **compute grid** (no network, no Postgres) that shards `data/raw/mdna.parquet` and calls `encode_documents` on each shard, writing `data/interim/mdna_embeddings_shard_*.npz`.
3. `jobs/merge_mdna_embeddings.sh` — merges the shards into `data/interim/mdna_embeddings.npy` + `data/interim/mdna_embeddings_keys.parquet` (the `(gvkey, fyear)` for each row, in the same order as the embeddings array).

This notebook picks up after step 3: it just loads the merged embeddings and keys, joins them onto the fundamentals panel, and reuses Module 10's `fit_gbm` unmodified.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from notebooks.m9_ols import build_dataset
from src.models.gbm import fit_gbm
from src.schema import FundamentalsSchema

FUNDAMENTALS_PATH = REPO_ROOT / "data" / "raw" / "fundamentals.parquet"
EMBEDDINGS_PATH = REPO_ROOT / "data" / "interim" / "mdna_embeddings.npy"
EMBEDDINGS_KEYS_PATH = REPO_ROOT / "data" / "interim" / "mdna_embeddings_keys.parquet"
RESULTS_DIR = REPO_ROOT / "results" / "m11"

## Load fundamentals and the grid-computed embeddings

Every row of `mdna.parquet` gets encoded by the grid job regardless of whether it has a matching fundamentals row, so "missing text" only happens on this side of the join: a fundamentals `(gvkey, fyear)` with no corresponding embedding. Those rows are dropped after the merge, with the count reported explicitly.

In [2]:
fundamentals = pd.read_parquet(FUNDAMENTALS_PATH)
FundamentalsSchema.validate(fundamentals, lazy=True)
fundamentals, features = build_dataset(fundamentals)

embeddings = np.load(EMBEDDINGS_PATH)
keys = pd.read_parquet(EMBEDDINGS_KEYS_PATH)
assert len(embeddings) == len(keys)

# Defensive re-sort: emb_change below assumes row i-1 is the immediately
# preceding fiscal year for the same firm, which only holds if the panel
# is ordered by (gvkey, fyear) — already true coming out of
# merge_embeddings.py, but asserting it here makes this cell correct on
# its own rather than relying on an upstream guarantee.
order = keys.sort_values(["gvkey", "fyear"]).index.to_numpy()
keys = keys.iloc[order].reset_index(drop=True)
embeddings = embeddings[order]

emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]

# Year-over-year MD&A language change: 1 - cosine_similarity(emb_t, emb_{t-1}).
# NaN whenever there's no immediately preceding fiscal year for the same
# firm (first year in the panel, or a gap year — e.g. from falling out of
# the S&P 500 and back in), so a multi-year gap is never silently scored
# as a one-year change.
normed = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
cos_sim = np.sum(normed[1:] * normed[:-1], axis=1)
emb_change = np.concatenate([[np.nan], 1.0 - cos_sim])

same_firm = keys["gvkey"].to_numpy()[1:] == keys["gvkey"].to_numpy()[:-1]
consecutive_year = keys["fyear"].to_numpy()[1:] == keys["fyear"].to_numpy()[:-1] + 1
valid = np.concatenate([[False], same_firm & consecutive_year])
emb_change[~valid] = np.nan

emb_df = pd.concat(
    [keys, pd.DataFrame(embeddings, columns=emb_cols), pd.Series(emb_change, name="emb_change")],
    axis=1,
)

n_before = len(fundamentals)
df = fundamentals.merge(emb_df, on=["gvkey", "fyear"], how="left")
df = df.dropna(subset=["emb_0"])
print(f"dropped {n_before - len(df)} of {n_before} rows with no matching MD&A embedding")
print(f"{len(df)} rows remain, of which {df['emb_change'].isna().sum()} have no year-over-year change (first year in panel or a gap)")

dropped 945 of 7461 rows with no matching MD&A embedding
6516 rows remain, of which 788 have no year-over-year change (first year in panel or a gap)


## Re-run Module 10's GBM: financials only, +text, +text+year-over-year change

Same time-based split as Module 10 (train: fyear <= 2020, test: fyear >= 2021). `test_eval` drops NaN in `emb_change` too (unlike the raw embedding columns, it's genuinely missing for a firm's first panel year or a gap year), so all three scenarios below are scored on the identical set of test rows.

In [3]:
features_with_emb = features + emb_cols
features_with_change = features_with_emb + ["emb_change"]

train = df[df["fyear"] <= 2020]
test = df[df["fyear"] >= 2021]
test_eval = test.dropna(subset=["target", *features_with_change])

print(f"train: {len(train)} rows (fyear {train['fyear'].min()}-{train['fyear'].max()})")
print(f"test:  {len(test)} rows (fyear {test['fyear'].min()}-{test['fyear'].max()})")
print(f"test rows usable for scoring: {len(test_eval)}")

train: 4704 rows (fyear 2010-2020)
test:  1812 rows (fyear 2021-2024)
test rows usable for scoring: 1261


In [4]:
_, preds_baseline = fit_gbm(train, test_eval, target="target", features=features)
rmse_baseline = mean_squared_error(test_eval["target"], preds_baseline) ** 0.5
r2_baseline = r2_score(test_eval["target"], preds_baseline)

_, preds_with_text = fit_gbm(train, test_eval, target="target", features=features_with_emb)
rmse_with_text = mean_squared_error(test_eval["target"], preds_with_text) ** 0.5
r2_with_text = r2_score(test_eval["target"], preds_with_text)

_, preds_with_change = fit_gbm(train, test_eval, target="target", features=features_with_change)
rmse_with_change = mean_squared_error(test_eval["target"], preds_with_change) ** 0.5
r2_with_change = r2_score(test_eval["target"], preds_with_change)

comparison = pd.DataFrame(
    [
        {"features": "financials only", "n_features": len(features), "test_rmse": rmse_baseline, "test_r2": r2_baseline},
        {"features": "financials + BERT text", "n_features": len(features_with_emb), "test_rmse": rmse_with_text, "test_r2": r2_with_text},
        {"features": "financials + BERT text + change", "n_features": len(features_with_change), "test_rmse": rmse_with_change, "test_r2": r2_with_change},
    ]
)
print(comparison.to_string(index=False))

fit_gbm: dropped 181 of 4704 train rows with NaN target


fit_gbm: dropped 181 of 4704 train rows with NaN target


fit_gbm: dropped 181 of 4704 train rows with NaN target


                       features  n_features  test_rmse  test_r2
                financials only          65   0.056632 0.480250
         financials + BERT text         833   0.057636 0.461661
financials + BERT text + change         834   0.058024 0.454384


In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RESULTS_DIR / "text_vs_no_text_comparison.csv", index=False)
print(f"comparison table written to {RESULTS_DIR / 'text_vs_no_text_comparison.csv'}")

comparison table written to /home/zhuwei/Projects/accy575/ACCY575-wrds-data-analysis/results/m11/text_vs_no_text_comparison.csv
